# Recurrent Neural Network (RNN)
"Finding Structure in Time" (Elman, 1990)

**Idea:** read a sequence one step at a time and carry a hidden state $h_t$ that summarizes everything seen so far. The **same weights** are applied at every step.

### RNN vs MLP vs Transformer
| | MLP | RNN | Transformer |
|---|---|---|---|
| Input | fixed-size vector | sequence of any length, one step at a time | whole sequence at once |
| Memory of the past | none | hidden state $h_t$ (fixed size) | attends directly to every past token |
| Order / position | - | implicit (processing order) | positional encoding |
| Training parallel over time? | - | **no**, step $t$ needs $h_{t-1}$ | yes |
| Cost per generated token | - | $O(1)$: update $h$ | $O(T)$: attend to all past tokens |
| Long-range memory | - | weak (vanishing gradients) | strong |

Shape flow: `idx (B, T) -> embed (B, T, E) -> loop t: h (B, H) -> stack (B, T, H) -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - tiny Shakespeare (character level)
~1.1M characters, vocab = 65 unique characters. Each training example is a random chunk of `block_size` characters; the target is the same chunk shifted by one (predict the next character).

The same data and loss are used in `rnn`, `lstm`, `gru`, `slm`, `moe`, `ssm`, so their val losses can be compared.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 64   # T: characters per sequence
batch_size = 64   # B

def get_batch(split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab size:", vocab_size, "| train chars:", len(train_data), "| val chars:", len(val_data))

## 2. RNN cell
One step: mix the current input with the previous hidden state, squash with tanh.

### Formula: RNN cell
$$h_t = \tanh\!\left(W_{xh} x_t + W_{hh} h_{t-1} + b_h\right)$$
- $x_t \in \mathbb{R}^{E}$: input at step $t$ (character embedding, $E = 64$)
- $h_{t-1} \in \mathbb{R}^{H}$: previous hidden state ($H = 256$), $h_0 = 0$
- $W_{xh} \in \mathbb{R}^{H \times E}$: input-to-hidden weights (`self.x2h`)
- $W_{hh} \in \mathbb{R}^{H \times H}$: hidden-to-hidden weights (`self.h2h`), shared across all steps
- $b_h$: bias
- $\tanh$: keeps $h_t$ in $(-1, 1)$ so it can't blow up over many steps

In [ ]:
class RNNCell(nn.Module):
    # x_t (B, E), h (B, H) -> h_new (B, H)
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.x2h = nn.Linear(input_size, hidden_size)               # W_xh, b_h
        self.h2h = nn.Linear(hidden_size, hidden_size, bias=False)  # W_hh

    def forward(self, x, h):
        return torch.tanh(self.x2h(x) + self.h2h(h))  # (B, H)

## 3. Character-level RNN language model
Embed each character, run the cell over time, and predict the next character from each $h_t$.

`forward` also takes and returns the state, so generation can continue from where it left off.

### Formula: Output layer
$$y_t = W_{hy} h_t + b_y, \qquad p(x_{t+1} \mid x_{\le t}) = \text{softmax}(y_t)$$
- $y_t \in \mathbb{R}^{V}$: logits over the vocab ($V = 65$)
- $W_{hy} \in \mathbb{R}^{V \times H}$: output weights (`self.head`)
- $h_t$ depends on all of $x_1 \dots x_t$ through the recurrence

In [ ]:
class CharRNN(nn.Module):
    # idx (B, T), h (B, H) or None -> logits (B, T, V), h (B, H)
    def __init__(self, vocab_size, emb_dim=64, hidden_size=256):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.cell = RNNCell(emb_dim, hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, h=None):
        B, T = idx.shape
        if h is None:
            h = torch.zeros(B, self.hidden_size, device=idx.device)
        x = self.embed(idx)                # (B, T, E)
        outs = []
        for t in range(T):                 # sequential: step t needs h from step t-1
            h = self.cell(x[:, t], h)      # (B, H)
            outs.append(h)
        out = torch.stack(outs, dim=1)     # (B, T, H)
        return self.head(out), h           # (B, T, V), (B, H)

model = CharRNN(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.3f}M")
logits, h = model(torch.zeros(2, 10, dtype=torch.long, device=device))
print(logits.shape, h.shape)

## 4. Training
Backpropagation through time (BPTT): unroll the loop over $T = 64$ steps and backprop through all of it. Gradient clipping is **required** for RNNs, see the formula below.

### Formula: Why RNN gradients vanish / explode
$$\frac{\partial h_T}{\partial h_k} = \prod_{t=k+1}^{T} \frac{\partial h_t}{\partial h_{t-1}} = \prod_{t=k+1}^{T} \text{diag}\!\left(1 - h_t^2\right) W_{hh}$$
- the gradient from step $T$ back to step $k$ is a product of $T - k$ Jacobians
- $1 - h_t^2 \le 1$: derivative of tanh
- if $\lVert W_{hh} \rVert < 1$ the product shrinks exponentially (**vanishing**: can't learn long-range dependencies)
- if $\lVert W_{hh} \rVert > 1$ it can grow exponentially (**exploding**: fixed by clipping)

### Formula: Next-token cross-entropy
$$\mathcal{L} = -\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T} \log p_\theta(x_{t+1} \mid x_{\le t})$$
- $B, T$: batch size and sequence length
- $x_{t+1}$: the true next character
- $p_\theta(\cdot \mid x_{\le t})$: softmax over the vocab from the logits at position $t$
- a model that guesses uniformly scores $\log 65 \approx 4.17$; lower is better

### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1, \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: all parameter gradients concatenated into one vector
- $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales the whole gradient, keeps its direction

In [ ]:
max_iters = 3000
eval_every = 500
lr = 3e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def loss_fn(model, x, y):
    logits, _ = model(x)                                              # (B, T, V)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # (B*T, V) vs (B*T,) -> scalar

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    x, y = get_batch("train")
    loss = loss_fn(model, x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 5. Generation
Feed the sampled character back in and carry $h$ forward. Each new character costs one cell step: $O(1)$ in sequence length.

### Formula: Sampling with temperature
$$p_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}, \qquad x_{t+1} \sim \text{Categorical}(p)$$
- $z_i$: logit of character $i$
- $\tau$: temperature (0.8); $\tau < 1$ sharpens (safer), $\tau > 1$ flattens (more random), $\tau \to 0$ = greedy argmax

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)       # (1, len(start))
    logits, state = model(idx)                                # read the prompt into the state
    out = list(start)
    for _ in range(n):
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)  # (1, V)
        next_id = torch.multinomial(probs, 1)                  # (1, 1)
        out.append(itos[next_id.item()])
        logits, state = model(next_id, state)                  # feed only the new char + carried state
    return "".join(out)

print(generate(model))

## Notes
- Expect val loss around ~1.6-1.7 here. The samples have the right shape (names, line breaks, short words) but little long-range structure.
- **Vanishing gradients** are the core weakness: the signal from 50+ steps back is multiplied by many Jacobians $< 1$. **LSTM** (next) fixes this with an additive cell state.
- `nn.RNN(emb_dim, hidden_size, batch_first=True)` is the same model with a fused CUDA kernel, much faster than our Python loop.